# 31. Acoustic response of the 27-gallon tote enclosure

Order-of-magnitude estimates for the HDPE storage totes that house the `HDI` Gems (and, at the
Centaur sites, the infraBSUs and electronics): air-cavity modes, lumped air compliance, lid-leak
Helmholtz resonance, panel bending modes and the low-frequency pressure transmission through the
walls. These are the values in the paper 1 supplement (Text S1 and Table S2); notebook 65 writes
the table from `tote_panel_modes.csv`.

In [1]:
import numpy as np
import pandas as pd
import artemis2_config as cfg

OUT = cfg.OUT_DIR / "31_tote_acoustics"
OUT.mkdir(parents=True, exist_ok=True)

# Tote and material (mean interior dimensions of the tapered tote)
L, W, H = 0.66, 0.43, 0.375          # m
MASS_KG = 2.77                        # 6.1 lb
c, rho_air = 343.0, 1.2               # m/s, kg/m^3
E, nu, rho_hdpe = 1.0e9, 0.4, 950.0   # HDPE: Pa, -, kg/m^3
H_RANGE = (2.0e-3, 2.6e-3)            # wall thickness range considered, m
CLAMPED = 1.85                        # clamped / simply supported frequency ratio
LEAK_AREA = 15e-4                     # ~1 mm slit around the ~1.5 m rim, m^2
LEAK_LENGTH = (2e-3, 5e-3)            # effective neck length, m

V = L * W * H
C_a = V / (rho_air * c**2)
area = 2 * (L * W + L * H + W * H)
h_mass = MASS_KG / area / rho_hdpe
print(f"V = {V:.3f} m^3, C_a = {C_a:.2e} m^3/Pa")
print(f"wall + base + lid area = {area:.2f} m^2 -> {MASS_KG/area:.1f} kg/m^2 -> h = {h_mass*1e3:.1f} mm")

V = 0.106 m^3, C_a = 7.54e-07 m^3/Pa
wall + base + lid area = 1.39 m^2 -> 2.0 kg/m^2 -> h = 2.1 mm


## Air-cavity modes (rigid rectangular cavity)

In [2]:
modes = sorted((c / 2 * np.sqrt((l / L)**2 + (m / W)**2 + (n / H)**2), (l, m, n))
               for l in range(3) for m in range(3) for n in range(3) if l + m + n)
for f, k in modes[:4]:
    print(f"{k}: {f:.0f} Hz")

(1, 0, 0): 260 Hz
(0, 1, 0): 399 Hz
(0, 0, 1): 457 Hz
(1, 1, 0): 476 Hz


## Lid leakage: Helmholtz resonance

In [3]:
f_H = [c / (2 * np.pi) * np.sqrt(LEAK_AREA / (V * le)) for le in LEAK_LENGTH]
print(f"f_H = {min(f_H):.0f}-{max(f_H):.0f} Hz for L_eff = {LEAK_LENGTH[0]*1e3:.0f}-{LEAK_LENGTH[1]*1e3:.0f} mm")

f_H = 92-145 Hz for L_eff = 2-5 mm


## Panel bending modes (Table S2)

Simply supported plate, $f_{mn} = (\pi/2)\sqrt{D/\rho_s}\,[(m/a)^2 + (n/b)^2]$. Each range runs from
simply supported edges with the thinnest wall to clamped edges with the thickest.

In [4]:
def plate_f(a, b, m, n, h, factor=1.0):
    D = E * h**3 / (12 * (1 - nu**2))
    return factor * np.pi / 2 * np.sqrt(D / (rho_hdpe * h)) * ((m / a)**2 + (n / b)**2)

PANELS = {"Base or lid": (L, W), "Long wall": (L, H), "Short wall": (W, H)}
rows = []
for panel, (a, b) in PANELS.items():
    for m, n in [(1, 1), (2, 1), (1, 2)]:
        rows.append(dict(panel=panel, a_m=a, b_m=b, mode=f"({m},{n})",
                         f_low_hz=plate_f(a, b, m, n, H_RANGE[0]),
                         f_high_hz=plate_f(a, b, m, n, H_RANGE[1], CLAMPED)))
panel_modes = pd.DataFrame(rows)
panel_modes.to_csv(OUT / "tote_panel_modes.csv", index=False)
panel_modes.pivot(index="panel", columns="mode", values=["f_low_hz", "f_high_hz"]).round(0)

f_low_hz             f_high_hz            
mode           (1,1) (1,2) (2,1)     (1,1) (1,2) (2,1)
panel                                                 
Base or lid      8.0  24.0  15.0      19.0  58.0  36.0
Long wall       10.0  31.0  17.0      23.0  75.0  40.0
Short wall      13.0  34.0  29.0      31.0  83.0  70.0

## Low-frequency transmission through the walls

Volume compliance of the lid as a simply supported plate under uniform pressure (double sine series),
compared with the air compliance: $p_\text{in}/p_\text{out} = C_w/(C_w + C_a)$.

In [5]:
def plate_compliance(a, b, h, nmax=199):
    D = E * h**3 / (12 * (1 - nu**2))
    k = np.arange(1, nmax + 1, 2)
    M, N = np.meshgrid(k, k)
    return 64 * a * b / (np.pi**8 * D) * np.sum(1 / (M**2 * N**2 * ((M / a)**2 + (N / b)**2)**2))

rows = []
for h in (H_RANGE[0], h_mass, H_RANGE[1]):
    Cw = plate_compliance(L, W, h)
    rows.append(dict(h_mm=h * 1e3, C_w=Cw, C_w_over_C_a=Cw / C_a, p_in_over_p_out=Cw / (Cw + C_a)))
lid = pd.DataFrame(rows)
lid

,h_mm,C_w,C_w_over_C_a,p_in_over_p_out
0,2.000000,0.000041,54.625891,0.982023
1,2.105111,0.000035,46.845019,0.979099
2,2.600000,0.000019,24.863856,0.961336


In [6]:
summary = pd.DataFrame([
    ("volume_m3", V), ("C_a_m3_per_Pa", C_a), ("wall_thickness_from_mass_mm", h_mass * 1e3),
    *[(f"cavity_mode_{k[0]}{k[1]}{k[2]}_hz", f) for f, k in modes[:4]],
    ("helmholtz_low_hz", min(f_H)), ("helmholtz_high_hz", max(f_H)),
    *[(f"lid_C_w_h{r.h_mm:.1f}mm_m3_per_Pa", r.C_w) for r in lid.itertuples()],
], columns=["quantity", "value"])
summary.to_csv(OUT / "tote_summary.csv", index=False)
summary

,quantity,value
0,volume_m3,1.064250e-01
1,C_a_m3_per_Pa,7.538313e-07
2,wall_thickness_from_mass_mm,2.105111e+00
3,cavity_mode_100_hz,2.598485e+02
4,cavity_mode_010_hz,3.988372e+02
5,cavity_mode_001_hz,4.573333e+02
6,cavity_mode_110_hz,4.760172e+02
7,helmholtz_low_hz,9.165434e+01
8,helmholtz_high_hz,1.449182e+02
9,lid_C_w_h2.0mm_m3_per_Pa,4.117871e-05
